<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/linearRegressionGoldnIDR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prediksi Harga Emas dalam Rupiah (IDR)**
Menggunakan Algoritma Regresi Linear Berbasis Data yfinance


---

Pada nootbook ini saya akan membangun model prediksi harga emas menggunakan pendekatan **Regresi Linear** dengan data historis yang diunduh langsung dari Yahoo Finance melalui **library yfinance**.

**Alur Kerja:**


1. Import dan Eksplorasi Data (EDA) menggunakan library ydata_profiling
2. Feature Engineering
3. Pre-Processing Data
4. Pelatihan Model
5. Evaluasi & Analisis Overfitting
6. Hyperparameter Tuning (Ridge & Lasso)
7. Percobaan Prediksi
8. Kesimpulan





In [80]:
# Install library yang diperlukan (Jalankan Sekali)

!pip install ydata-profiling
!pip install yfinance

# **INFORMASI DATASET**

Dataset ini berisi data historis **Harga Emas Dunia dan Indeks USD/IDR** yang dikumpulkan oleh Yahoo Finance dalam rentang waktu **2023 hingga sekarang**.

**Sumber Data**: Yahoo Finance - Gold Price, Indeks USD/IDR (2023-Sekarang)

| Ticker | Deskripsi | Satuan |
|--------|-----------|--------|
| `GC=F` | Gold Futures (Harga Emas Dunia) | USD/troy oz |
| `USDIDR=X` | Kurs USD terhadap Rupiah | IDR per 1 USD |

Rentang data 1 Januari 2023 - 29 Juni 2026

Harga emas akhirnya akan dikonversi ke IDR dengan rumus:
> Harga Emas (IDR/gram) = Harga Emas (USD/troy oz) x Kurs USD/IDR / 31.1035








In [81]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from ydata_profiling import ProfileReport
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# digunakan untuk memberitahu dan mengatur peringatan fitur yg usang
import warnings
warnings.filterwarnings('ignore')

**# Langkah 1 Mengambil Data Dari Library YFinance**



1.   Ambil data harga emas dan indeks nilsi tukar USD/IDR
2.   Gunakan ticker symbol yang tersedia di yfinance.
3. Gunakan dictionary untuk memudahkan pengambilan data.
4. Tentukan rentang pengambilan data, saya menggunakan data historis dari januari 2023 - sekarang.




In [82]:
# untuk konversi ounce ke gram
TROY_OUNCE_TO_GRAM = 31.1035   # 1 troy ounce = 31.1035 gram
pd.set_option('display.float_format', lambda x: '%.2f' % x)
ticker_map = {
    "GC=F"    : "gold",
    "USDIDR=X": "usdidr",
}

START = "2023-01-01"
END   = datetime.today().strftime("%Y-%m-%d")

raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True,
    group_by    = 'ticker'
)

print(f"\n✅ Data Emas   : {raw.shape[0]} baris, {raw["GC=F"].shape[1]} kolom")
print(f"✅ Data USD/IDR: {raw.shape[0]} baris, {raw["USDIDR=X"].shape[1]} kolom")

print("Preview Data Harga Emas berdasarkan Data Yahoo Finance")
display(raw["GC=F"].())
print("Preview Data Kurs USD /IDR berdasarkan Data Yahoo Finance")
display(raw["USDIDR=X"].head())


[*********************100%***********************]  2 of 2 completed


✅ Data Emas   : 906 baris, 5 kolom
✅ Data USD/IDR: 906 baris, 5 kolom
Preview Data Harga Emas berdasarkan Data Yahoo Finance


Price,Open,High,Low,Close,Volume
Date,,,,,
2023-01-02,NaN,NaN,NaN,NaN,NaN
2023-01-03,1836.20,1839.70,1836.20,1839.70,29.00
2023-01-04,1845.60,1859.10,1845.60,1852.80,25.00
2023-01-05,1855.20,1855.20,1834.80,1834.80,24.00
2023-01-06,1838.40,1868.20,1835.30,1864.20,26.00


Preview Data Kurs USD /IDR berdasarkan Data Yahoo Finance


Price,Open,High,Low,Close,Volume
Date,,,,,
2023-01-02,15508.80,15588.70,15463.90,15508.80,0.00
2023-01-03,15550.00,15638.80,15514.90,15550.00,0.00
2023-01-04,15571.80,15627.50,15553.50,15571.80,0.00
2023-01-05,15560.00,15640.90,15558.50,15560.00,0.00
2023-01-06,15620.00,15692.10,15558.10,15620.00,0.00


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [83]:
raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True
)

# df = raw["Close"].rename(columns=ticker_map).add_suffix("_close")
print(f"\nShape setelah download: {df.shape}")
print(df)
print(f"Rentang tanggal awal  : {df.index.min().date()} s.d. {df.index.max().date()}")

[*********************100%***********************]  2 of 2 completed


Shape setelah download: (1269, 5)
Ticker      gold_close  usdidr_close  gold_close_idr_per_oz  \
Date                                                          
2023-01-02     1839.70      15508.80            28531538.24   
2023-01-03     1839.70      15550.00            28607334.24   
2023-01-04     1852.80      15571.80            28851431.44   
2023-01-05     1834.80      15560.00            28549488.76   
2023-01-06     1864.20      15620.00            29118803.24   
...                ...           ...                    ...   
2026-06-19     4224.10      17794.40            75165328.43   
2026-06-20     4224.10      17794.40            75165328.43   
2026-06-21     4224.10      17794.40            75165328.43   
2026-06-22     4181.90      17788.00            74387635.46   
2026-06-23     4129.90      17862.00            73768272.06   

Ticker      gold_close_idr_per_gram  harga_besok_per_gram  
Date                                                       
2023-01-02               

In [84]:
# TAHAP 2: PENYELARASAN TIMEZONE (SHIFT +1 HARI)
GLOBAL_TICKERS = list(ticker_map.keys())

for ticker_name in GLOBAL_TICKERS:
  col = f"{ticker_name}_close"
  if col in df.columns:
    df[col] = df[col].shift(1)

In [85]:
# TAHAP 3: REINDEX KE KALENDER HARIAN PENUH

full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

print(f"Shape telah reindex: {df.shape}")
print(f"Rentang tanggal baru: {df.index.min().date()} s.d. {df.index.max().date()}")

Shape telah reindex: (1269, 5)
Rentang tanggal baru: 2023-01-02 s.d. 2026-06-23


In [86]:
# TAHAP 4: FORWARD FILL (FFILL) -> HARGA CLOSE HARI TERAKHIR

price_cols = [c for c in df.columns if c.endswith("_close")]
df[price_cols] = df[price_cols].ffill().bfill()

missing_after = df[price_cols].isnull().sum()
print(f"\nJumlah missing value setelah forward fill:\n{missing_after}")
print()



Jumlah missing value setelah forward fill:
Ticker
gold_close      0
usdidr_close    0
dtype: int64



In [87]:
# # feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df['usdidr_close']

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]
print(df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]].tail(3).to_string())

Ticker      gold_close  usdidr_close  gold_close_idr_per_oz  gold_close_idr_per_gram
Date                                                                                
2026-06-21     4224.10      17794.40            75165328.43               2416619.62
2026-06-22     4181.90      17788.00            74387635.46               2391616.23
2026-06-23     4129.90      17862.00            73768272.06               2371703.25


In [88]:
# 2. Suruh YData buat "cetak biru" analisisnya
profile = ProfileReport(df, title="Laporan Analisis Data Melzzz")

# 3. Simpan hasilnya jadi halaman web (HTML)!
profile.to_file("laporan_web_kamu.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:00<00:00, 110.91it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [89]:
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
#Setelah mendapatkan dataclean . hapus field antara X dan Y

df['harga_besok_per_gram'] = df['gold_close_idr_per_gram'].shift(-1)
df = df.dropna()
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram", "harga_besok_per_gram"]]
dataclean.tail()

x = dataclean[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
y = dataclean["harga_besok_per_gram"]

In [90]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = model.score(X_test, y_test)
print(f"R2 Score: {score} ")
print(f"R2 Score: {r2_score(y_test, y_pred)}")
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

R2 Score: 0.9540838558800842 
R2 Score: 0.9540838558800842
Mean Squared Error: 1866443477.5288029
